# InterX Government Intelligence Engine
## Google Colab 실행 가이드

정부지원사업 공고 자동 수집 · 점수화 · Google Sheets 업로드 파이프라인

---
**실행 순서**: 셀을 위에서 아래로 순서대로 실행하세요 (`Shift+Enter`)

| 단계 | 내용 |
|------|------|
| 1 | 저장소 클론 + 패키지 설치 |
| 2 | Colab 환경 패치 (asyncio) |
| 3 | Google Drive 마운트 (선택) |
| 4 | 인증 설정 (service_account.json) |
| 5 | 환경변수 설정 |
| 6 | 파이프라인 실행 |

## 1단계 — 저장소 클론 + 패키지 설치

> **약 2~3분 소요** (Playwright chromium 설치 포함)

In [ ]:
# ── 저장소 클론 ──────────────────────────────────────────────────────────────
import os

REPO_DIR = '/content/interx-gov-intelligence'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/KimDoojin2/interx-gov-intelligence.git {REPO_DIR}
else:
    print('이미 클론됨 — 최신 코드로 업데이트')
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('\n작업 디렉토리:', os.getcwd())

In [ ]:
# ── 패키지 설치 ──────────────────────────────────────────────────────────────
# Colab에는 requests·pandas·matplotlib 등이 이미 설치되어 있어 빠르게 완료됩니다
!pip install -q \
    beautifulsoup4 lxml \
    gspread google-auth google-auth-oauthlib \
    PyYAML python-dateutil python-dotenv \
    scikit-learn openpyxl \
    wordcloud \
    nest-asyncio \
    playwright \
    pypdf python-docx olefile

print('\n패키지 설치 완료')

In [ ]:
# ── Playwright 크로미움 설치 ─────────────────────────────────────────────────
# bizinfo(기업마당), kiat, dicia 사이트에 필요
# 일반 사이트만 수집할 경우 이 셀은 건너뛰어도 됩니다
!playwright install chromium
!playwright install-deps chromium
print('\nPlaywright 크로미움 설치 완료')

## 2단계 — Colab 환경 패치

Colab은 이미 이벤트 루프가 실행 중이므로 `nest_asyncio`로 충돌을 방지합니다.

In [ ]:
# ── asyncio 충돌 방지 (Colab 필수) ───────────────────────────────────────────
import nest_asyncio
nest_asyncio.apply()

# ── sys.path 등록 ─────────────────────────────────────────────────────────────
import sys
REPO_DIR = '/content/interx-gov-intelligence'
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')

# ── 한국어 폰트 설치 (분석 차트에 사용) ──────────────────────────────────────
!apt-get install -q fonts-nanum
import matplotlib
matplotlib.font_manager._load_fontmanager(try_read_cache=False)

import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print('환경 패치 완료')

## 3단계 — Google Drive 마운트 (선택)

> **건너뛰어도 됩니다.** Drive를 마운트하면 수집 결과(SQLite·CSV·분석 PNG)가 Drive에 영구 저장됩니다.
> 마운트하지 않으면 Colab 세션 종료 시 데이터가 사라집니다.

In [ ]:
# ── Google Drive 마운트 (선택) ────────────────────────────────────────────────
USE_DRIVE = True   # Drive 사용 원하지 않으면 False 로 변경

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    import os
    DRIVE_DATA = '/content/drive/MyDrive/interx_data'
    os.makedirs(DRIVE_DATA, exist_ok=True)

    # data/ 디렉토리를 Drive로 심볼릭 링크
    DATA_LOCAL = f'{REPO_DIR}/data'
    if not os.path.islink(DATA_LOCAL):
        if os.path.exists(DATA_LOCAL):
            import shutil
            shutil.move(DATA_LOCAL, DRIVE_DATA + '_backup')
        os.symlink(DRIVE_DATA, DATA_LOCAL)
        print(f'data/ → {DRIVE_DATA} 심볼릭 링크 완료')
    else:
        print('이미 Drive 연결됨')
else:
    import os
    os.makedirs(f'{REPO_DIR}/data', exist_ok=True)
    print('로컬 저장 모드 (세션 종료 시 데이터 삭제됨)')

## 4단계 — Google Sheets 인증 설정

두 가지 방법 중 하나를 선택하세요.

- **방법 A** (권장): Colab 파일 업로드로 `service_account.json` 업로드
- **방법 B**: Drive에 저장된 파일 경로 지정
- **방법 C**: Sheets 없이 콘솔 출력만 (`--no-sheets` 옵션)

> `service_account.json` 발급 방법:
> GCP Console → IAM & Admin → Service Accounts → Keys → Add Key → JSON

In [ ]:
# ── 방법 A: 파일 직접 업로드 ─────────────────────────────────────────────────
import os

SA_PATH = f'{REPO_DIR}/service_account.json'

if not os.path.exists(SA_PATH):
    from google.colab import files
    print('service_account.json 파일을 선택하세요:')
    uploaded = files.upload()

    for filename, content in uploaded.items():
        with open(SA_PATH, 'wb') as f:
            f.write(content)
    print(f'인증 파일 저장 완료: {SA_PATH}')
else:
    print(f'기존 인증 파일 사용: {SA_PATH}')

In [ ]:
# ── 방법 B: Drive에 저장된 파일 사용 (방법 A 사용 시 이 셀 건너뜀) ───────────
# Drive 마운트 후 파일 경로를 아래에 입력하세요

import os, shutil
DRIVE_SA = '/content/drive/MyDrive/service_account.json'  # 여기 경로 수정
SA_PATH  = f'{REPO_DIR}/service_account.json'

if os.path.exists(DRIVE_SA) and not os.path.exists(SA_PATH):
    shutil.copy(DRIVE_SA, SA_PATH)
    print(f'Drive에서 인증 파일 복사 완료: {SA_PATH}')
elif os.path.exists(SA_PATH):
    print(f'인증 파일 이미 존재: {SA_PATH}')
else:
    print(f'Drive 경로를 확인하세요: {DRIVE_SA}')

## 5단계 — 환경변수 설정

여기서 Google Sheets 이름, Telegram/Slack 알림 토큰을 설정합니다.

In [ ]:
import os

# ── 필수: Google Sheets 스프레드시트 이름 ─────────────────────────────────────
# 서비스 계정 이메일을 해당 스프레드시트의 편집자로 추가해야 합니다
os.environ['INTERX_SHEET_NAME'] = 'InterX 영업기회 CRM'   # ← 실제 스프레드시트 이름으로 변경

# ── 선택: Telegram 알림 ───────────────────────────────────────────────────────
# os.environ['TELEGRAM_BOT_TOKEN'] = 'your-bot-token'
# os.environ['TELEGRAM_CHAT_ID']   = 'your-chat-id'

# ── 선택: Slack 알림 ─────────────────────────────────────────────────────────
# os.environ['SLACK_WEBHOOK_URL'] = 'https://hooks.slack.com/services/...'

# ── 선택: Claude API (L3 공고 자동 요약) ──────────────────────────────────────
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'

print('환경변수 설정 완료')
print(f"  시트 이름: {os.environ.get('INTERX_SHEET_NAME')}")
print(f"  Telegram : {'설정됨' if os.environ.get('TELEGRAM_BOT_TOKEN') else '미설정'}")
print(f"  Slack    : {'설정됨' if os.environ.get('SLACK_WEBHOOK_URL') else '미설정'}")

## 6단계 — 파이프라인 실행

아래 셀 중 **하나만** 실행하세요.

In [ ]:
# ════════════════════════════════════════════════════════════
#  [A] 테스트 실행 (Mock 데이터 — 실제 크롤링 없음, 빠름)
#      처음 실행할 때 이걸 먼저 확인하세요
# ════════════════════════════════════════════════════════════
%cd /content/interx-gov-intelligence
!python run_engine.py --dry-run --no-sheets --no-alert

In [ ]:
# ════════════════════════════════════════════════════════════
#  [B] 빠른 실제 수집 (JS 불필요 사이트 + 목록만, 약 2~5분)
#      Playwright 미설치 시 이 옵션 사용
# ════════════════════════════════════════════════════════════
%cd /content/interx-gov-intelligence
!python run_engine.py \
    --sites iris,bipa,uipa,gicon,ttp,gjtp,gbtp,jntp,jbtp,jejutp \
    --max-pages 2 \
    --no-detail \
    --no-alert \
    --no-sheets

In [ ]:
# ════════════════════════════════════════════════════════════
#  [C] 전체 수집 (상세 페이지 포함, 약 10~20분)
#      Google Sheets 업로드까지 포함
# ════════════════════════════════════════════════════════════
%cd /content/interx-gov-intelligence
!python run_engine.py \
    --max-pages 3 \
    --no-alert

In [ ]:
# ════════════════════════════════════════════════════════════
#  [D] 특정 사이트만 수집
#      사이트 키: bizinfo iris kiat smba nipa innopolis
#                bipa uipa gicon ttp dicia gjtp gbtp jntp jbtp jejutp
# ════════════════════════════════════════════════════════════
%cd /content/interx-gov-intelligence
!python run_engine.py \
    --sites bizinfo,iris,kiat,smba,nipa \
    --max-pages 2 \
    --no-alert

## 7단계 — 결과 확인

In [ ]:
# ── 자동 생성된 분석 차트 표시 ────────────────────────────────────────────────
import glob
import os
from IPython.display import Image, display

charts = sorted(
    glob.glob('/content/interx-gov-intelligence/data/analysis/dashboard_*.png'),
    reverse=True
)

if charts:
    print(f'가장 최근 분석 차트: {os.path.basename(charts[0])}')
    display(Image(charts[0], width=1100))
else:
    print('분석 차트 없음 — 파이프라인을 먼저 실행하세요')

In [ ]:
# ── SQLite에서 수집된 공고 목록 확인 ─────────────────────────────────────────
import sqlite3, pandas as pd, glob, os

db_files = glob.glob('/content/interx-gov-intelligence/data/*.db')

if not db_files:
    print('DB 파일 없음 — 파이프라인을 먼저 실행하세요')
else:
    db_path = db_files[0]
    print(f'DB: {db_path}')

    con = sqlite3.connect(db_path)

    # 테이블 목록
    tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)
    print('\n테이블 목록:', tables['name'].tolist())

    # 최근 수집 공고
    try:
        df = pd.read_sql("""
            SELECT site, title, priority_grade, deadline_date, budget
            FROM notices
            ORDER BY rowid DESC
            LIMIT 30
        """, con)
        print(f'\n최근 수집 공고 ({len(df)}건):')
        display(df)
    except Exception as e:
        print(f'notices 테이블 조회 실패: {e}')

    con.close()

In [ ]:
# ── 등급 분포 차트 직접 그리기 ────────────────────────────────────────────────
import sqlite3, pandas as pd, matplotlib.pyplot as plt
import glob

db_files = glob.glob('/content/interx-gov-intelligence/data/*.db')
if not db_files:
    print('DB 없음')
else:
    con = sqlite3.connect(db_files[0])
    try:
        df = pd.read_sql("SELECT priority_grade, site FROM notices", con)
        con.close()

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # 등급 분포
        grade_cnt = df['priority_grade'].value_counts().reindex(['A','B','C','D']).fillna(0)
        colors = ['#2ecc71','#3498db','#f39c12','#e74c3c']
        ax1.pie(grade_cnt, labels=[f'{g}등급 ({int(v)}건)' for g,v in grade_cnt.items()],
                colors=colors, autopct='%1.0f%%', startangle=90)
        ax1.set_title('공고 등급 분포', fontsize=14, fontweight='bold')

        # 사이트별 수집 건수
        site_cnt = df['site'].value_counts().head(10)
        ax2.barh(site_cnt.index[::-1], site_cnt.values[::-1], color='#3498db')
        ax2.set_xlabel('수집 건수')
        ax2.set_title('사이트별 수집 건수 (Top 10)', fontsize=14, fontweight='bold')
        for i, v in enumerate(site_cnt.values[::-1]):
            ax2.text(v + 0.1, i, str(v), va='center')

        plt.tight_layout()
        plt.savefig('/content/grade_distribution.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f'\n총 {len(df)}건 수집')
    except Exception as e:
        con.close()
        print(f'오류: {e}')

In [ ]:
# ── L3 강공고 목록 출력 ───────────────────────────────────────────────────────
import sqlite3, pandas as pd, glob

db_files = glob.glob('/content/interx-gov-intelligence/data/*.db')
if not db_files:
    print('DB 없음')
else:
    con = sqlite3.connect(db_files[0])
    try:
        df = pd.read_sql("""
            SELECT site, title, deadline_date, budget, recommended_solution
            FROM notices
            WHERE l3_strong = 'Y'
            ORDER BY rowid DESC
        """, con)
        print(f'L3 강공고: {len(df)}건')
        pd.set_option('display.max_colwidth', 60)
        display(df)
    except Exception as e:
        print(f'오류: {e}')
    finally:
        con.close()

In [ ]:
# ── 결과 파일 Drive로 복사 (Drive 마운트 시) ─────────────────────────────────
import os, shutil, glob

if os.path.exists('/content/drive'):
    SAVE_DIR = '/content/drive/MyDrive/interx_results'
    os.makedirs(SAVE_DIR, exist_ok=True)

    # 분석 PNG 복사
    for f in glob.glob('/content/interx-gov-intelligence/data/analysis/*.png'):
        shutil.copy(f, SAVE_DIR)

    # DB 복사
    for f in glob.glob('/content/interx-gov-intelligence/data/*.db'):
        shutil.copy(f, SAVE_DIR)

    print(f'Drive 저장 완료: {SAVE_DIR}')
    print('저장된 파일:')
    for f in os.listdir(SAVE_DIR):
        print(f'  {f}')
else:
    print('Drive 미마운트 — 3단계에서 Drive 연결 후 실행하세요')

## 자동 스케줄링 (선택)

Colab을 열어두면 매일 지정 시간에 자동 실행할 수 있습니다.

> **주의**: Colab 무료 버전은 최대 12시간 연속 실행 제한이 있습니다.

In [ ]:
# ── 매일 오전 9시 자동 실행 스케줄러 ─────────────────────────────────────────
import schedule, time, subprocess, threading
from datetime import datetime

def run_pipeline():
    print(f'\n[{datetime.now().strftime("%Y-%m-%d %H:%M")}] 파이프라인 시작...')
    result = subprocess.run(
        ['python', 'run_engine.py', '--max-pages', '3', '--no-alert'],
        cwd='/content/interx-gov-intelligence',
        capture_output=True, text=True
    )
    print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
    if result.returncode != 0:
        print('오류:', result.stderr[-500:])
    print(f'[{datetime.now().strftime("%Y-%m-%d %H:%M")}] 완료')

# 매일 오전 9:00 실행 (시간 수정 가능)
schedule.every().day.at('09:00').do(run_pipeline)

print('스케줄러 시작 — 매일 09:00 실행')
print('중단하려면 커널 중단 (Ctrl+C 또는 런타임 → 실행 중단)')

def _run_scheduler():
    while True:
        schedule.run_pending()
        time.sleep(30)

t = threading.Thread(target=_run_scheduler, daemon=True)
t.start()
print('백그라운드 스케줄러 실행 중...')

---

## 트러블슈팅

| 증상 | 원인 | 해결 |
|------|------|------|
| `No module named 'interx_engine'` | sys.path 미등록 | 2단계 셀 재실행 |
| `playwright._impl._errors.Error` | chromium 미설치 | 1단계 playwright 설치 셀 재실행 |
| Google Sheets 업로드 실패 | 서비스 계정 권한 없음 | 스프레드시트 → 공유 → 서비스 계정 이메일 편집자 추가 |
| `invalid_grant` 오류 | service_account.json 만료 | GCP에서 새 키 발급 후 4단계 재실행 |
| 수집 0건 | 사이트 HTML 구조 변경 | `--sites` 옵션으로 특정 사이트만 지정 후 확인 |
| 한글 깨짐 | 폰트 미설치 | 2단계 셀에서 `apt-get install fonts-nanum` 확인 |

### 사이트별 수집 방식

| 방식 | 사이트 |
|------|--------|
| **requests** (빠름) | iris, bipa, uipa, gicon, ttp, gjtp, gbtp, jntp, jbtp, jejutp, smba, nipa, innopolis |
| **Playwright** (JS 렌더링) | bizinfo(기업마당), kiat, dicia |

Playwright 설치가 번거로우면 `--sites` 옵션으로 requests 사이트만 선택하세요.